# 📓 MapBiomas Fire Monitor — Pipeline (M0-M7)


## 📘 Documentación y Guía del Usuario
Expanda esta sección para entender la arquitectura del dato, reglas de nomenclatura y fluxo de trabalho.

#### 📌 Introducción y Contexto de Uso

Este pipeline de código constituye el núcleo de procesamiento automatizado para el **Monitoramiento de Cicatrices de Fuego de MapBiomas**. Su objetivo principal es extraer, procesar, clasificar y publicar datos satelitales a nivel regional o nacional con alta trazabilidad.

*   📥 **Entradas Globales (Inputs):** Colecciones de imágenes brutas desde Google Earth Engine (Sentinel-2, Landsat), polígonos vectoriales para entrenamiento (muestras / *samples*), y mapas auxiliares de cobertura de suelo (LULC).
*   📤 **Salidas Globales (Outputs):** Chunks y mosaicos optimizados (COGs) en Google Cloud Storage (GCS), pesos de modelos de redes neuronales (DNN), matrices de clasificación, e ImageCollections versionadas pre-oficiales en GEE.
*   🔄 **Alternativas de Ejecución:** La arquitectura modular permite **ejecución mixta**.
    *   **Ejecución Local (Sugerida para M1, M2 e M5):** Máxima estabilidad al descargar y ensamblar mosaicos con GDAL a través de I/O de disco sostenido.
    *   **Google Colab (Sugerida para M3 y M4):** Acceso inmediato a recursos de RAM/GPU, facilitando recolección de muestras ágil y entrenamiento de redes.

> **Nota:** Tanto el disco del PC local como el almacenamiento temporal de Colab actúan como **espacio efímero**. La persistencia ocurre siempre en el **Google Cloud Storage (GCS) Bucket**.


#### 🚦 Ciclo de Vida del Dato y Reglas de Nomenclatura

| Etapa | Peso | Inputs → Outputs | Regla de Nomenclatura | Ejemplo |
| :--- | :--- | :--- | :--- | :--- |
| **M1: Export** | **Leve** | **IN:** Colecciones GEE<br>**OUT:** Chunks GCS | `[sensor]_fire_[país]_[año][mes]` | `s2_fire_peru_2408` |
| **M2: Mosaic** | **Medio** | **IN:** Chunks GCS<br>**OUT:** Mosaicos COG (GCS) | Semilla M1 + sufijo banda | `s2_fire_peru_2408_nir_cog.tif` |
| **M3: Samples (Gateway)** | **Leve** | **IN:** Mosaicos (M2)<br>**OUT:** Polígonos (GCS/Asset) | `[col]_[sat_ref]_[reg]`| `samples_sentinel2_r01` |
| **M4: Train** | **Medio** | **IN:** Muestras M3 + Mosaicos M2<br>**OUT:** DNN (GCS) | `[ver]_[sensor]_[reg]` | `v1_1_sentinel2_r01` |
| **M5: Classify**| **Pesado**| **IN:** Modelo M4 + Mosaicos M2<br>**OUT:** Raster (GCS) | `klass_[país]_[reg]_[mod]_[yymm]`| `klass_peru_r01_v1_2408` |
| **M6: Filters** | **Leve** | **IN:** DOY M5 + LULC<br>**OUT:** Raster Filtrado (GCS)| `filt_[país]_[reg]_[mod]_[yymm]`| `filt_peru_r01_v1_2408` |
| **M7: Versioner**| **Leve** | **IN:** Candidatos M6<br>**OUT:** Asset PRE-OFICIAL | `[colección]_v[X]` | `peru_fire_col1_v1` |


### 📂 Arquitectura de Datos e Relacionamento (M1-M7)

O monitor opera um fluxo circular de sincronização entre três ambientes:

| Ambiente | Componentes Principais | Papel no Ciclo |
| :--- | :--- | :--- |
| **🌍 Google Earth Engine** | ImageCollections (Mosaicos) / Assets | Fonte de Brutos e Destino Final |
| **☁️ Google Cloud Storage** | Chunks / COGs / Models / Samples | Persistência e Área de Trabalho Central |
| **⚡ Cache Local (Temp)** | VRT Stack / Tiles / NumPy Arrays | Processamento I/O de Alta Velocidade |

#### 🧭 Mapa de Persistência (Onde encontrar os dados)

| Etapa | Extensão | Path Principal no Cloud Storage (GCS) | 
| :--- | :--- | :--- |
| **M1: Export** | `.tif` | `library_images/{sensor}/monthly/chunks/{yyyy}/{mm}/` |
| **M2: Mosaic** | `.tif` | `library_images/{sensor}/monthly/cog/{yyyy}/{mm}/` |
| **M3: Samples** | `.shp` | `library_samples/{anual,monthly}/{ano}/` |
| **M4: Train** | `.pb / .json` | `library_images/{sensor}/model/{model_name}/` |
| **M5: Classify** | `.tif` | `library_images/{sensor}/monthly/classifications/raw_versions/v{v}/` |
| **M7: Public** | Asset IC | `projects/mapbiomas-public/assets/{country}/fire/monitor/` |

#### 🏷️ Regras de Nomenclatura Padrão
*   **Fragmentos (M1):** `{sensor}_fire_{pais}_{yy}{mm}_{banda}.tif`
*   **Mosaico COG (M2):** `{sensor}_fire_{pais}_{yy}{mm}_{banda}_cog.tif`
*   **Classificação (M5):** `class_{sensor}_{pais}_{modelo}_{yymm}_v{v}.tif`

---

#### 🔄 Retroalimentación y Tolerancia a Fallos

*   **Restantes:** El botón "Seleccionar Restantes" marcará solo los ítems **pendientes** en GCS.
*   **Sobrescritura:** Remarcar manualmente un ítem con bandera de éxito (verde) indica la intención de reemplazar el archivo.
*   **Modo Edición (`EDIT_MODE`):** Si es activado en `M0`, la UI expone botones de "Eliminar" en GCS.


## ⚙️ [M0] — Configuración de Ambiente (Escolha uma Rota)


### > Opción A: Inicialización Google Colab


In [ ]:
# M0.1a — Preparación del entorno Colab (Clonar repo)
import os
if not os.path.exists("peru-fire"):
    !git clone https://github.com/mapbiomas/peru-fire
%cd peru-fire/mapbiomas_fire_monitor/version_01/src/core


#### > Instalación de Dependencias Vitales (GDAL)
El monitor utiliza utilitarios de sistema de **GDAL** para ensamblar mosaicos COG de alto rendimiento. Esta celda debe ejecutarse una vez por sesión en Colab.

In [1]:
# ⬇️ Instalar GDAL Binaries e dependências Python
!apt-get update -qq && apt-get install -y -qq gdal-bin python3-gdal
!pip install -q earthengine-api gcsfs rasterio scipy tqdm


'apt-get' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


^C


In [ ]:
# Autenticación en Google Cloud / GEE
from google.colab import auth
auth.authenticate_user()


### > Opción B: Inicialización Local


**🛠️ Requisitos Local (GDAL / Conda)**

Si recibes un error de `Faltam dependências vitais` ejecutando localmente, asegúrate de tener os binários do GDAL instalados.
*   **Conda:** `conda install -c conda-forge gdal` (Recomendado)
*   **Nota:** `pip install gdal` no es suficiente para as herramientas de mosaico.


In [2]:
# M0.1b — Configuración local de rutas
import sys, os
REPO_ROOT = os.path.abspath("..")
SRC_PATH  = os.path.join(REPO_ROOT, "src", "core")
if SRC_PATH not in sys.path: sys.path.insert(0, SRC_PATH)

# Requisito Local: GDAL (gdalbuildvrt, gdal_translate) deve estar no PATH
# Dica: Use 'conda install -c conda-forge gdal' ou OSGeo4W no Windows
# En consola: (1) earthengine authenticate (2) gcloud auth application-default login


## 🚀 Rota Sequencial (Fluxo Padrão de Processamento)


In [5]:
# M0.2 — Inicialización del Monitor
import sys, os

# Auto-ajuste de rotas (Garante que o Python encontre os scripts)
REPO_ROOT = os.path.abspath("..")
SRC_PATH  = os.path.join(REPO_ROOT, "src", "core")
if os.path.exists(SRC_PATH) and SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
    print(f"✅ Rota de scripts configurada: {SRC_PATH}")

COUNTRY   = "peru"
EDIT_MODE = True   # True → mostra Deletar + Rewriter na toolbar

from M0_auth_config import set_country, authenticate, print_config, set_global_opts, set_edit_mode
set_country(COUNTRY)
set_global_opts(sensor='sentinel2', periodicity='monthly')
set_edit_mode(EDIT_MODE)
authenticate()
print_config()

✅ GEE já autenticado.
✅ Autenticação GCS/ADC configurada.
  🔥 MapBiomas Fire Monitor — Pipeline
  País         : PERU (peru)
  Projeto GEE  : mapbiomas-peru
  Bucket GCS   : gs://mapbiomas-fire
  Base Path    : sudamerica/peru/monitor
  Sensor (Fluxo): SENTINEL2
  Periodicidade : MONTHLY
  Modo Edição   : ✅ ATIVO
  Sensor EE     : COPERNICUS/S2_SR_HARMONIZED
  Bandas espec. : ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
  Bandas modelo : ['red', 'nir', 'swir1', 'swir2']
  Tile size     : 0.83° (~92km)
  LULC mask     : classes [26, 22, 33, 24]
  LULC Asset    : projects/mapbiomas-public/assets/peru/collection2/mapbiomas_peru_collection2_integration_v1


### [M1] — Despacho de Exportación (GEE → Bucket)

**Peso:** 🔵 Leve | **Acción:** Selección tabular y despacho de tareas GEE.


In [13]:
import importlib
import M1_export_logic
import M1_export_ui

# Força o recarregamento de ambos os módulos
importlib.reload(M1_export_logic)
importlib.reload(M1_export_ui)

# Recria a interface e tenta o export novamente
ui_exporter = M1_export_ui.run_ui(years=[2025,2026])



>>> M1_export_ui inicializando (v6.0 ASCII) <<<
M1: [1/3] Iniciando dados...


M1: [2/3] Construindo interface...
M1: [3/3] Renderizacao concluida.


In [14]:
from M1_export_ui import start_export
start_export(ui_exporter)


### [M2] — Ensamblaje Nacional (COG)

**Peso:** 🟡 Medio | **Acción:** Unión de fragmentos mediante GDAL (VRT -> COG).


In [8]:
import importlib
import M2_mosaic_ui
importlib.reload(M2_mosaic_ui)
from M2_mosaic_ui import run_ui, start_assemble

# Dispara a renderizacao segura
ui_assembler = run_ui(years=[2025,2026])



>>> M2_mosaic_ui inicializando (v6.0 ASCII) <<<

>>> M2_mosaic_ui inicializando (v6.0 ASCII) <<<
M2: [1/3] Iniciando dados do GCS...


M2: [2/3] Construindo interface...
M2: [3/3] Renderizacao concluida.


In [ ]:
from M2_mosaic_ui import start_assemble
start_assemble(ui_assembler) # Si obtiene NameError, verifique la celda anterior


### [M3] — Coleta de Amostras (GEE Toolkit Gateway)


In [15]:
from M3_sample_ui import show_toolkit_links
show_toolkit_links()



 🖍️  M3 - COLETA DE AMOSTRAS (GEE TOOLKIT GATEWAY)

  A etapa de coleta de amostras é realizada exclusivamente
  através da interface JavaScript no Google Earth Engine.

  🔗 1. Acesso Rápido ao Código-Fonte do Toolkit (GitHub):
     https://github.com/mapbiomas/mapbiomas-fire/tree/main/peru/src/gee/src/core/M3_toolkit.js

  🔗 2. Documentação e Padrões de Uso:
     https://github.com/mapbiomas/mapbiomas-fire/blob/main/peru/FIRE_MONITOR_STANDARDS.md#mockup-ascii-m32-gee-toolkit-gateway




True

### [M4] — Entrenamiento del Modelo (DNN)

**Peso:** 🟡 Medio | **Acción:** Configuración de hiperparámetros y entrenamiento.


In [ ]:
from M4_model_trainer import run_ui as run_trainer
ui_trainer = run_trainer()


In [18]:
from M4_model_trainer import start_training
start_training(ui_trainer)


### [M5] — Clasificación Regional

**Peso:** 🔴 Pesado | **Acción:** Inferencia masiva regional.


In [ ]:
# Preset de modelos por región
PRESET_MODELS = {
  "v1_deep_sentinel2_peru": ["loreto_norte", "san_martin"],
  "v2_exp_sentinel2_peru":  ["ucayali_sur", "madre_de_dios"]
}
from M5_classifier import run_ui as run_classifier
ui_classifier = run_classifier(PRESET_MODELS)


In [ ]:
from M5_classifier import start_classification
start_classification(ui_classifier)


### [M6] — Aplicación de Filtros Físicos y LULC

**Peso:** 🟢 Leve | **Acción:** Aplicación de máscaras y filtros morfológicos focalizados.


In [ ]:
# Preset de filtros por región
PRESET_FILTERS = {
    "peru_r1": {
        "mask_classes": [26, 33, 24],
        "open_filter": 3,
        "close_filter": 3
    }
}
from M6_publisher import run_ui as run_filters
ui_filters = run_filters(PRESET_FILTERS)


In [ ]:
from M6_publisher import start_filtering
start_filtering(ui_filters)


### [M7] — Curaduría y Colección Pre-Oficial

**Peso:** 🟣 Morado | **Acción:** Selección de ganadores y exportación a GEE Asset.


In [ ]:
# Preset de votos (variante ganadora por región)
PRESET_VOTES = {
    "peru_r1": "filt_peru_r1_v1_sentinel2_2408"
}
from M7_curator import run_ui as run_curator
ui_curator = run_curator(PRESET_VOTES)


In [ ]:
from M7_curator import start_curation
start_curation(ui_curator)
